In [1]:
# Ignore all other GPUs except this one
!export CUDA_VISIBLE_DEVICES=0

In [2]:
import numpy as np         
import matplotlib.pyplot as plt     
from matplotlib.animation import FuncAnimation          
import torch      
import torch.nn as nn    
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from torch.utils.data import TensorDataset, DataLoader        
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 
import time      
from scipy.ndimage import uniform_filter1d    
import pandas as pd
import pickle 
import os
from IPython.display import HTML

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

Assume we have $\phi$ and $n_e$ measurements.\
Want to reconstruct $\bold{E}$ and $\bold{v_e}$

In [18]:
##Plasma Parmeters in normalized units##
kB_Te = 1.0  # Electron temperature, chosen for normalization (k_B T_e in eV)
epsilon_0 = 1
lambda_D = 1.0  # Debye length, chosen as unit length scale
omega_p = 1.0  # Electron plasma frequency, chosen as unit time scale
m_e = 1 #normalize electron mass to 1 for simplicity
q_e = 1 #normalize electorn charge to 1 for simplicity 
v_th = np.sqrt(kB_Te/m_e)
gamma = 3

# Domain dimensions in normalized units
x_min, x_max = -5 * lambda_D, 5 * lambda_D  # Normalized spatial domain. normalize in terms of skin depth
Lx = x_max - x_min 
t_min, t_max = 0, 10 * (2 * np.pi / omega_p)  # Normalized time domain. normalize in terms of omega_p
tau = t_max - t_min

nt = 200 #number of time steps
nx = 200 #number of spatial poitns
dt = .01 # time steps in plasma periods omeg_p**-1
dx = Lx/nx 
#normalized wavenumber
k=2 * np.pi/Lx

#sample amplitudes
phi0 = 1
n0 = 1.0  # Equilibrium electron density, normalized

#Dispersion relation for langmuir waves#
omega = np.sqrt(omega_p**2 + gamma*(k**2)*(v_th**2))

In [5]:
def analytical_solution(x, t, phi0):
    
    phi = phi0*np.cos(k*x-omega*t)
    Ex = -k*phi0*np.sin(k*x-omega*t)
    ne = n0 * np.cos(k*x - omega*t)
    ve = (q_e/m_e*omega)*Ex
    
    return phi,Ex,ne,ve

def generate_data(nx, nt, Lx, tau, phi0):

    x = np.linspace(0, Lx, nx)
    t = np.linspace(0, tau, nt)

    x_arr, t_arr =  np.meshgrid(x,t, indexing='xy')

    phi, Ex, ne, ve = analytical_solution(x_arr,t_arr, phi0)
    
    return x_arr.flatten(), t_arr.flatten(), phi.flatten(), Ex.flatten(), ne.flatten(), ve.flatten()

def sparse_measurements(x, t, phi, ne, num_samples):

    indices = np.random.choice(x.shape[0], num_samples, replace=False)

    return x[indices], t[indices], phi[indices], ne[indices]

def collocation_points(nx, nt, Lx, tau):

    x = np.linspace(0, Lx, nx)
    t = np.linspace(0, tau, nt)

    x_coll, t_coll = np.meshgrid(x, t, indexing='xy')

    return x_coll.flatten(), t_coll.flatten()

In [6]:
Nx, Nt = 250, 250

X_flat, T_flat, Phi_flat, Ex_flat, ne_flat, ve_flat= generate_data(Nx, Nt, Lx, tau, phi0)

x_sparse, t_sparse, phi_sparse, ne_sparse = sparse_measurements(X_flat, T_flat, \
                                                                Phi_flat, ne_flat, \
                                                                num_samples=10000)
x_coll, t_coll = collocation_points(Nx,Nt,Lx,tau)

In [7]:
class Sin(nn.Module):

    def __init__(self):
        super(Sin, self).__init__()

    def forward(self, x):
        return torch.sin(x)
    
class Tanh(nn.Module): 
    
    def __init__(self): 
        super(Tanh,self).__init__()
    
    def forward(self, x): 
        return torch.tanh(x)
    
class Swish(nn.Module): 
    
    def __init__(self): 
        super(Swish, self).__init__()
    
    def forward(self, x): 
        return (x/(1+torch.exp(x)))
    
class Sigmoid(nn.Module): 
    
    def __inint__(self): 
        super(Sigmoid, self).__init__init()
    def forward(self, x): 
        return (1/(1+torch.exp(x)))


class PINN(nn.Module):
    
    def __init__(self, Lx, tau): 
        
        super(PINN, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(2,100),
            Tanh(), 
            nn.Linear(100,100),
            Tanh(),
            nn.Linear(100,100),
            Tanh(),
            nn.Linear(100,4)
        )

        self.Lx = Lx
        self.tau = tau
        
    def forward(self, x,  t):
        
        # This operation normalizes the input coordinates to be contained within the range [-1, 1].
        
        x = 2.0*(x / self.Lx) - 1
        t = 2.0*(t / self.tau) - 1
        
        inputs = torch.cat([x,t], dim = 1)
        
        return self.net(inputs)

We are assuming an electrostatic oscillation\
Equation Constraints:\
(1) $mn_e(\frac{\partial\bold{v_e}}{\partial t} + (\bold{v_e}\cdot\nabla)\bold{v_e}) + en_e\bold{E} = 0$\
(2) $\frac{\partial n_{e}}{\partial t} + \nabla \cdot (n_e \bold{v_e}) = 0$\
(3) $\nabla \cdot \bold{E} + \frac{en_e}{\epsilon_0}=0$\
(4) $\bold{E} + \nabla\phi = 0$


In [36]:
def pinn_loss(model, x_sparse, t_sparse, phi_sparse, ne_sparse, means, stds, x_col, t_col):
    
    # Data loss measuring loss of Bdots
    
    sparse_preds = model(x_sparse, t_sparse)
    
    phi_sparse_preds = sparse_preds[:,0].reshape(-1,1)#potential measurements
    phi_sparse_preds = (phi_sparse_preds - means)/stds
    phi_loss = torch.mean(torch.square(phi_sparse_preds - phi_sparse))
    ne_sparse_preds = sparse_preds[:,2].reshape(-1,1)#density measurements
    ne_sparse_preds = (phi_sparse_preds - means)/stds
    ne_loss = torch.mean(torch.square(ne_sparse_preds-ne_sparse))    
    loss_data = phi_loss+ne_loss
    
    
    #Physics loss from collocation points
    col_preds = model(x_col, t_col) * stds + means
    phi = col_preds[:,0]
    E = col_preds[:,1]
    ne = col_preds[:,2]
    ve = col_preds[:,3]
    
    
    E = E.reshape(-1,1)
    phi = phi.reshape(-1,1)
    
    #time derivatives and gradients
    phi_x = torch.autograd.grad(phi, x_col, grad_outputs=torch.ones_like(phi), create_graph=True, retain_graph=True)[0]
    E_x = torch.autograd.grad(E, x_col, grad_outputs=torch.ones_like(E), create_graph=True, retain_graph=True)[0]
    ne_t = torch.autograd.grad(ne, t_col, grad_outputs=torch.ones_like(ne), create_graph=True, retain_graph=True)[0]
    ne_tt = torch.autograd.grad(ne_t, t_col, grad_outputs=torch.ones_like(ne_t), create_graph=True, retain_graph=True)[0]
    ne_x = torch.autograd.grad(ne, x_col, grad_outputs=torch.ones_like(ne), create_graph=True, retain_graph=True)[0]
    ne_xx = torch.autograd.grad(ne_x, x_col, grad_outputs=torch.ones_like(ne_x), create_graph=True, retain_graph=True)[0]
    ve_t = torch.autograd.grad(ve, t_col, grad_outputs=torch.ones_like(ve), create_graph=True, retain_graph=True)[0]
    ve_x = torch.autograd.grad(ve, x_col, grad_outputs=torch.ones_like(ve), create_graph=True, retain_graph=True)[0]
    
    
    #physics constrain loss
    momentum = m_e*ne*(ve_t + (ve)*ve_x) + q_e*ne*E
    momentum = torch.mean(momentum**2)
    dens_cont = ne_t + ne*ve_x + ne_x*ve
    dens_cont = torch.mean(dens_cont**2)
    gauss = E_x + (q_e*ne/epsilon_0)
    gauss = torch.mean(gauss**2)
    phi_E = E + phi_x
    phi_E = torch.mean(phi_E**2)
    # den_pressure = ne_tt + (omega_p**2)*ne - gamma*(v_th**gamma)*ne_xx
    # den_pressure = torch.mean(den_pressure**2)
    
    
    loss_physics = momentum + dens_cont + gauss + phi_E #+ den_pressure

    return loss_data, loss_physics

In [37]:
# Convert data to PyTorch tensors
x_sparse = torch.tensor(x_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
t_sparse = torch.tensor(t_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
phi_sparse = torch.tensor(phi_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
ne_sparse = torch.tensor(ne_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)


x_coll = torch.tensor(x_coll.flatten(), dtype=torch.float32).reshape(-1,1).requires_grad_()
t_coll = torch.tensor(t_coll.flatten(), dtype=torch.float32).reshape(-1,1).requires_grad_()

means = torch.stack((torch.mean(phi_sparse, 0).detach(), torch.mean(phi_sparse, 0).detach(), torch.mean(ne_sparse, 0).detach(), torch.mean(phi_sparse, 0).detach()), dim=1)
stds = torch.stack((torch.std(phi_sparse, 0).detach(), torch.std(phi_sparse, 0).detach(), torch.mean(ne_sparse, 0).detach(), torch.mean(phi_sparse, 0).detach()), dim=1)


means[:,0] = 0.0
stds[:,0] = 1.0
means[:,1] = 0.0
stds[:,1] = 1.0
means[:,2] = 0.0
stds[:,2] = 1.0
means[:,3] = 0.0
stds[:,3] = 1.0



print('Means = ', means)
print('Stds = ', stds)

# Create DataLoaders
batch_size = int(x_coll.size()[0] // 50)

Means =  tensor([[0., 0., 0., 0.]])
Stds =  tensor([[1., 1., 1., 1.]])


/tmp/ipykernel_119734/2631928360.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_sparse = torch.tensor(x_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
/tmp/ipykernel_119734/2631928360.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_sparse = torch.tensor(t_sparse.flatten(), dtype=torch.float32, requires_grad=True).reshape(-1,1)
/tmp/ipykernel_119734/2631928360.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  phi_sparse = torch.tensor(phi_sparse.flatten(), dtype=torch.float32, requires_grad=True).re

In [38]:
start_time = time.time()
model = PINN(Lx, tau)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50, verbose=True)

num_epochs = 1000
n_batches = 50


hist = {
    'data_loss': [],
    'physics_loss': [],
    'total_loss': [],
    'learning_rate': []
}

for epoch in range(num_epochs):
    
    epoch_sm_loss = 0
    epoch_phys_loss = 0
    epoch_total_loss = 0
    num_batches = 0

    for i in range(n_batches):

        n_coll = x_coll.shape[0]
        i_idxs = np.random.choice(n_coll, size = n_coll, replace = False)

        x_col_batch = x_coll[i_idxs[i::n_batches],:]
        t_col_batch = t_coll[i_idxs[i::n_batches],:]
        
        optimizer.zero_grad()
        
        sm_loss, phys_loss = pinn_loss(model, x_sparse, t_sparse, phi_sparse, ne_sparse, means, stds, x_col_batch, t_col_batch)
     
        total_loss = sm_loss + phys_loss
        
        total_loss.backward()
        optimizer.step()

        epoch_sm_loss += sm_loss.item()
        epoch_phys_loss += phys_loss.item()
        epoch_total_loss += total_loss.item()
        
        num_batches += 1

    # Calculate average losses for the epoch
    avg_sm_loss = epoch_sm_loss / num_batches
    avg_phys_loss = epoch_phys_loss / num_batches
    avg_total_loss = epoch_total_loss / num_batches

    # Step the scheduler
    scheduler.step(avg_total_loss)
    
    # Record history
    hist['data_loss'].append(avg_sm_loss)
    hist['physics_loss'].append(avg_phys_loss)
    hist['total_loss'].append(avg_total_loss)
    hist['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Total Loss: {avg_total_loss:.4f}, "
              f"Sample Measurement Loss: {avg_sm_loss:.4f}, Physics Loss: {avg_phys_loss:.4f}, "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")
end_time = time.time() 
print("--- %s mins ---" % str(round((end_time-start_time)/60,2)))

Epoch 10/1000, Total Loss: 0.9964, Sample Measurement Loss: 0.9964, Physics Loss: 0.0000, LR: 0.001000
Epoch 20/1000, Total Loss: 0.9946, Sample Measurement Loss: 0.9945, Physics Loss: 0.0001, LR: 0.001000
Epoch 30/1000, Total Loss: 0.9937, Sample Measurement Loss: 0.9936, Physics Loss: 0.0002, LR: 0.001000
Epoch 40/1000, Total Loss: 0.9934, Sample Measurement Loss: 0.9932, Physics Loss: 0.0002, LR: 0.001000
Epoch 50/1000, Total Loss: 0.9931, Sample Measurement Loss: 0.9929, Physics Loss: 0.0002, LR: 0.001000
Epoch 60/1000, Total Loss: 0.9921, Sample Measurement Loss: 0.9917, Physics Loss: 0.0004, LR: 0.001000
Epoch 70/1000, Total Loss: 0.9919, Sample Measurement Loss: 0.9914, Physics Loss: 0.0005, LR: 0.001000
Epoch 80/1000, Total Loss: 0.9913, Sample Measurement Loss: 0.9907, Physics Loss: 0.0006, LR: 0.001000
Epoch 90/1000, Total Loss: 0.9913, Sample Measurement Loss: 0.9906, Physics Loss: 0.0007, LR: 0.001000
Epoch 100/1000, Total Loss: 0.9902, Sample Measurement Loss: 0.9894, Phys

In [1]:
# Plot the loss history
plt.figure(figsize=(12, 8))
plt.semilogy(hist['data_loss'], label='Data Loss')
plt.semilogy(hist['physics_loss'], label='Physics Loss')
plt.semilogy(hist['total_loss'], label='Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss History')
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'plt' is not defined

In [ ]:
with torch.no_grad():

    X_reconstruct = torch.tensor(X_flat.reshape(-1,1), requires_grad=False, dtype=torch.float32, device=device)
    T_reconstruct = torch.tensor(T_flat.reshape(-1,1), requires_grad=False, dtype=torch.float32, device=device)

    # T_reconstruct_phdt = T_reconstruct + 0.5*dt*torch.ones(T_reconstruct.size(), device=T_reconstruct.device)
    # T_reconstruct_mhdt = T_reconstruct - 0.5*dt*torch.ones(T_reconstruct.size(), device=T_reconstruct.device)

    phi_pred, E_pred, ne_pred, ve_pred= (model(X_reconstruct, T_reconstruct)).T
    # _, _, Bz_reconstruct_phdt = (model(X_reconstruct, T_reconstruct_phdt)*stds + means).T
    # _, _, Bz_reconstruct_mhdt = (model(X_reconstruct, T_reconstruct_mhdt)*stds + means).T

    # Bz_dot_pred = (Bz_reconstruct_phdt - Bz_reconstruct_mhdt) / dt

# Plot the field components at t = 0

ext = [0, Lx, 0, tau]

plt.figure(figsize=(16,12))

i_t = 0
thistime = (tau / Nt)*i_t

plt.subplot(3, 4, 1)
plt.title(rf'$E_x(t = {thistime:.2f}$' + r'$\, \omega^{-1}) \,\, [E_0]$')
plt.imshow(E_flat.reshape(Nx,Nt)[:,:,i_t], origin='lower', aspect='auto', extent=ext, vmin=-phi0, vmax=phi0, cmap='RdBu')
plt.xlabel(r'$x \, [L_x]$')
plt.ylabel(r'$y \, [L_y]$')
plt.colorbar()

plt.subplot(3, 4, 2)
plt.title(rf'$E_y(t = {thistime:.2f}$' + r'$\, \omega^{-1}) \,\, [E_0]$')
plt.imshow(Ey_flat.reshape(Nx,Nt)[:,:,i_t], origin='lower', aspect='auto', extent=ext, vmin=-phi0, vmax=phi0, cmap='PuOr')
plt.xlabel(r'$x \, [L_x]$')
plt.ylabel(r'$y \, [L_y]$')
plt.colorbar()



plt.tight_layout()
plt.show()

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)